# 03 — Transfer Learning (ResNet18, Two-Phase Fine-Tuning)
Runs Phase 1 (frozen backbone) then Phase 2 (unfrozen layer3+layer4, 10x lower backbone LR), then builds the ablation table: baseline vs phase1 vs phase2.

In [2]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: C:\Users\Akshat Agarwal\Downloads\satellite-landuse-changedetection (1)\satellite-landuse-changedetection


In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
from src.training import train_phase1, train_phase2
train_phase1.main()


Using device: cuda


phase1 epochs:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
train_phase2.main()


In [ ]:
import yaml, torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from src.data.datasets import EuroSATDataset
from src.data.transforms import get_eval_transform
from src.data.splits import spatial_block_split
from src.models.baseline_cnn import BaselineCNN
from src.models.resnet_classifier import build_resnet18
from src.training.engine import evaluate, load_checkpoint
from src.evaluation.metrics import per_class_f1, macro_f1, plot_confusion_matrix

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classes = cfg['data']['eurosat_classes']

ds = EuroSATDataset(cfg['paths']['data_raw'], transform=get_eval_transform(cfg['data']['image_size']), download=False)
splits = spatial_block_split(ds, val_fraction=cfg['data']['val_fraction'], test_fraction=cfg['data']['test_fraction'], seed=cfg['seed'])
val_loader = DataLoader(Subset(ds, splits['val']), batch_size=cfg['data']['batch_size'])
criterion = nn.CrossEntropyLoss()

results = {}
for name, ckpt, builder in [
    ('baseline_cnn', 'baseline_cnn.pt', lambda: BaselineCNN(num_classes=len(classes), image_size=cfg['data']['image_size'])),
    ('phase1_frozen', 'resnet18_phase1.pt', lambda: build_resnet18(num_classes=len(classes), pretrained=False)),
    ('phase2_finetuned', 'resnet18_eurosat_finetuned.pt', lambda: build_resnet18(num_classes=len(classes), pretrained=False)),
]:
    model = builder()
    model = load_checkpoint(model, f"{cfg['paths']['checkpoints']}/{ckpt}", device).to(device)
    m = evaluate(model, val_loader, criterion, device)
    results[name] = m['macro_f1']
    print(f'{name}: macro-F1 = {m["macro_f1"]:.4f}')
    if name == 'phase2_finetuned':
        plot_confusion_matrix(m['labels'], m['preds'], classes,
                               save_path='../outputs/confusion_matrices/phase2_cm.png',
                               title='Phase 2 (Fine-tuned) Confusion Matrix')

print('\nAblation table:', results)
